In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from scipy.stats import mode

In [ ]:
# 1. Load and preprocess images
image_size = (32, 32)
path = r"C:\Users\Asus\Desktop\apple-leaf-disease-classification\data\dataset_split\train"

data = []
labels = []

MAX_IMAGES_PER_CLASS = 200

label_names = sorted(os.listdir(path))  # Sorted for consistent encoding
for label in label_names:
    folder_path = os.path.join(path, label)
    count = 0
    for file in os.listdir(folder_path)[:MAX_IMAGES_PER_CLASS]:
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, image_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        data.append(img.flatten())  # Convert to 1D feature vector
        labels.append(label)
        count += 1

data = np.array(data)
labels = np.array(labels)

# 2. Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)
n_clusters = len(le.classes_)
print(f"Classes: {le.classes_}")
print(f"Number of clusters: {n_clusters}")
print(f"Total images loaded: {len(data)}")

In [ ]:
# 3. Reduce dimensions with PCA before K-Means (speeds up and improves clustering)
pca = PCA(n_components=50, random_state=42)
data_pca = pca.fit_transform(data)
print(f"Explained variance ratio (50 components): {pca.explained_variance_ratio_.sum():.2f}")

# 4. Fit K-Means
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(data_pca)

# 5. Map each cluster to the most frequent true label (majority vote)
cluster_to_label = {}
for cluster in range(n_clusters):
    mask = cluster_labels == cluster
    majority = mode(labels_encoded[mask], keepdims=True).mode[0]
    cluster_to_label[cluster] = majority

y_pred = np.array([cluster_to_label[c] for c in cluster_labels])

acc = accuracy_score(labels_encoded, y_pred)
print(f"\nK-Means Clustering Accuracy: {acc * 100:.2f}%")

# Confusion Matrix
cm = confusion_matrix(labels_encoded, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
print("\nClassification Report:\n")
print(classification_report(labels_encoded, y_pred, target_names=le.classes_))

In [ ]:
# 6. Predict function for new image
def predict_image_class(image_path, kmeans_model, pca_model, cluster_map, image_size=(32, 32)):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not load image")
    img = cv2.resize(img, image_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pca = pca_model.transform(img.flatten().reshape(1, -1))
    cluster = kmeans_model.predict(img_pca)[0]
    return le.inverse_transform([cluster_map[cluster]])[0]

# Example prediction
new_image_path = r"C:\Users\Asus\Desktop\apple-leaf-disease-classification\test.jpg"
try:
    result = predict_image_class(new_image_path, kmeans, pca, cluster_to_label)
    print(f"Predicted Disease Category: {result}")
except Exception as e:
    print("Prediction Error:", e)